In [1]:
!pip install torch torchvision scikit-learn matplotlib seaborn -q

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import time
import copy

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

classes = train_dataset.classes

print("Classes:")
print(classes)

print("\nTraining samples:", len(train_dataset))
print("Testing samples :", len(test_dataset))

Classes:
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Training samples: 50000
Testing samples : 10000


In [5]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully.")

DataLoaders created successfully.


In [6]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    10
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=10, bias=True)


In [7]:
def train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    epochs=5
):

    history = {
        "train_loss": [],
        "train_acc": []
    }

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        start_time = time.time()

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()
            optimizer.step()

            running_loss += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = correct / total

        history["train_loss"].append(
            epoch_loss
        )

        history["train_acc"].append(
            epoch_acc
        )

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {epoch_loss:.4f} "
            f"Accuracy: {epoch_acc:.4f} "
            f"Time: {time.time()-start_time:.1f}s"
        )

    return history

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Training classification head...")

history_stage1 = train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    epochs=5
)

Training classification head...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/5] Loss: 1.4525 Accuracy: 0.4937 Time: 3228.5s
Epoch [2/5] Loss: 1.2700 Accuracy: 0.5519 Time: 3198.1s


In [ ]:
# Unfreeze the last ResNet block
for param in model.layer4.parameters():
    param.requires_grad = True

# Keep earlier layers frozen
for param in model.layer1.parameters():
    param.requires_grad = False

for param in model.layer2.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = False

# Classification layer remains trainable
for param in model.fc.parameters():
    param.requires_grad = True


optimizer = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=0.0001,
    weight_decay=1e-4
)

print("Fine-tuning last ResNet block...")

history_stage2 = train_model(
    model,
    train_loader,
    criterion,
    optimizer,
    epochs=5
)

In [ ]:
train_losses = (
    history_stage1["train_loss"]
    + history_stage2["train_loss"]
)

train_accs = (
    history_stage1["train_acc"]
    + history_stage2["train_acc"]
)

epochs_range = range(
    1,
    len(train_losses) + 1
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    epochs_range,
    train_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")
plt.grid(alpha=0.3)

plt.show()


plt.figure(figsize=(10, 5))

plt.plot(
    epochs_range,
    train_accs,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Accuracy")
plt.title("Training Accuracy")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            _, predictions = torch.max(
                outputs,
                1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    return (
        np.array(all_labels),
        np.array(all_predictions)
    )


y_true, y_pred = evaluate_model(
    model,
    test_loader
)

accuracy = accuracy_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print(
    f"Test Accuracy: {accuracy:.4f}"
)

print(
    f"Weighted F1 Score: {f1:.4f}"
)

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=classes
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("CIFAR-10 Confusion Matrix")

plt.show()

In [ ]:
# Get a batch from test data
images, labels = next(
    iter(test_loader)
)

images_gpu = images.to(device)

model.eval()

with torch.no_grad():

    outputs = model(images_gpu)

    _, predictions = torch.max(
        outputs,
        1
    )


# Display images
plt.figure(figsize=(15, 8))

for i in range(12):

    image = images[i].numpy()

    # Undo normalization
    image = (
        image.transpose(1, 2, 0)
        * np.array([0.229, 0.224, 0.225])
        + np.array([0.485, 0.456, 0.406])
    )

    image = np.clip(
        image,
        0,
        1
    )

    plt.subplot(3, 4, i + 1)

    plt.imshow(image)

    true_label = classes[
        labels[i].item()
    ]

    predicted_label = classes[
        predictions[i].item()
    ]

    plt.title(
        f"True: {true_label}\nPred: {predicted_label}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "resnet18_cifar10_transfer_learning.pth"
)

print(
    "Model saved successfully!"
)

In [ ]:
print("=" * 60)
print("TRANSFER LEARNING PROJECT RESULTS")
print("=" * 60)

print(f"Model           : ResNet18")
print(f"Dataset         : CIFAR-10")
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples    : {len(test_dataset)}")
print(f"Test Accuracy   : {accuracy * 100:.2f}%")
print(f"Weighted F1     : {f1:.4f}")

print("=" * 60)